<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/09-serving-inference/02-inference-performance.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 推理性能（概念篇）

**目标：** 理解 LLM 服务器吞吐与延迟背后的那几个杠杆（连续批处理、KV cache、量化，以及吞吐与延迟之间的取舍），并会做粗略估算来给一个部署定容量。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

## 为什么要有这一篇

[第 01 篇](01-serving-frameworks.zh.ipynb) 说过，服务化框架的全部职责就是在*昂贵的* GPU 上高效应答*大量*请求，而且正是这一个约束驱动了每一项优化。本篇讲的就是那些优化。它区分的是「我用了 vLLM」和能够回答*为什么 vLLM 快*以及*你的服务需要多少张 GPU*——这正是 AI 系统面试会追问的问题。

和第 01 篇一样，它是**概念优先**的：心智模型和定容量的算术在任何 runtime 上都能跑（CPU 就行，反正只是算术）。末尾有一个可选、带围栏的附录，在 GPU 上实测一条真实的批处理曲线。

> **⭐ 关键结论 ——** LLM 推理是**受内存带宽限制，而不是受算力限制**。GPU 大部分时间都花在*搬运权重和 cache*，而不是做数学。下面几乎每一个性能杠杆，本质上都是在耍花招少搬点内存，或者把原本闲着等内存的 GPU 重新用起来。

## 先分清大家总在混为一谈的两个数字

性能从来不是一个数字，而是两个，且彼此取舍：

- **延迟（latency）：***单个*请求要多久。再细分成 **TTFT**（time to first token，首个 token 的时间，也就是你 prompt 的 *prefill*）和 **TPOT**（time per output token，每个输出 token 的时间，对应每一步 *decode*）。聊天界面就靠 TTFT 活命；批处理任务只在乎总量。
- **吞吐（throughput）：** 服务器*在所有人身上每秒*能产出多少 token/请求。它决定你的 GPU 账单。

这两个在打架：主要的吞吐杠杆（下面的批处理）会*抬高*单请求延迟。你不说清**对谁而言**，「快」就毫无意义：是那一个等待的用户，还是整个机队的 token 账单。

> **🔵 面试信号 ——** 当被要求「让推理更快」时，高手的做法是反问：*「你要延迟还是吞吐？这个场景是交互式还是批处理？」* 先点明取舍再动手优化，是老练的标志。

## 杠杆 1 —— 连续批处理（最重要的一个）

一次只跑一个请求的 GPU 几乎全程闲置：一步 decode 在等内存时，几乎没碰到它的算力。**批处理**把许多请求的 decode 步骤*合在一起*跑，于是一次昂贵的权重读取就服务了整个 batch。这就是吞吐的倍增器。

朴素（「静态」）批处理要等整个 batch 跑完才开下一批，于是一个长生成就把所有人卡住。**连续批处理**（vLLM 的招牌特性）则相反，*每一步*都把已完成的请求换出去、把新请求换进来，让 GPU 始终保持满载。同样的硬件，在真实流量下能拿到数倍的吞吐。

下面这个单元格是对*它为什么能赢*的简化模型：不用 GPU，只有闲置时间的算术。

In [ ]:
# 玩具模型：一步 GPU 处理 1 个请求和 N 个请求的代价相同（直到某个上限），
# 因为代价主要在把权读取一遍。所以批处理把这次读取摊薄了。
STEP_MS = 20          # 一步 decode：几乎是固定的、受内存带宽限制，与 batch size 无关（直到饱和）
TOKENS_PER_REQ = 200  # 每个请求的输出 token 数

def throughput(batch_size, step_ms=STEP_MS):
    # N 个请求同步 decode：在 tokens*step_ms 的墙钟时间里产出 N*tokens 个 token。
    reqs = batch_size
    wall_s = (TOKENS_PER_REQ * step_ms) / 1000
    return reqs / wall_s, (TOKENS_PER_REQ * batch_size) / wall_s  # req/s, tok/s

print(f"{'batch':>5} {'req/s':>8} {'tok/s':>9}   note")
for b in [1, 4, 16, 32]:
    r, t = throughput(b)
    note = "GPU mostly idle" if b == 1 else "amortizing the weight read across the batch"
    print(f"{b:>5} {r:>8.1f} {t:>9.0f}   {note}")
print("\n-> same GPU, same per-step cost: throughput scales with batch until memory (the KV cache) runs out.")

> **⚠️ 生产现实 ——** 批处理正是为什么在负载下，*你自己的*延迟反而会*升高*，尽管服务器「更快」了。你的请求现在要和别人共享每一步 GPU。这就是吞吐/延迟取舍的具体体现，也是为什么交互式服务会限制 batch size 或预留余量，而批处理管线会把它拉满。

## 杠杆 2 —— KV cache（它给 batch 封顶）

当模型生成第 50 个 token 时，它需要已经为第 1–49 个 token 算出来的 key/value。每一步都重算的话复杂度会变成平方级；于是服务器把它们**缓存**在 GPU 显存里，这就是 **KV cache**。它随着 batch 里*每一个*请求的*每一个* token 一起增长。

所以给 batch size 封顶的通常是 KV cache，而不是算力：

- 这就是为什么算力充足的 GPU 仍然没法批更多：它是**KV cache 显存不够了**，而不是算力不够。
- 这正是 **vLLM 的 PagedAttention** 所优化的地方：像操作系统的虚拟内存那样用固定页来管理 cache，于是碎片不会白白浪费它。这就是第 01 篇里「单张 GPU 能服务多得多的并发请求」背后的机制。
- 这就是为什么**长上下文很贵**：cache 大小与序列长度成正比。上下文翻倍，每个请求的 cache 大致翻倍，能批的数量就大致减半。

下面这个单元格把 KV cache 算成一个数字，而不是一种感觉。

In [ ]:
# KV cache 大小 = 2（K 和 V）* layers * seq_len * hidden * bytes_per_val。
# 下面的数字大致是一个 fp16 的 7B 模型。这个计算就是给你的 batch 上限封顶的算式。
def kv_cache_gb(seq_len, layers=32, hidden=4096, bytes_per_val=2):
    bytes_ = 2 * layers * seq_len * hidden * bytes_per_val
    return bytes_ / 1e9

GPU_KV_BUDGET_GB = 12  # 例如一张 24GB 的 GPU，扣掉权重后大约一半留给 cache

print(f"{'context':>8} {'KV/req':>9} {'max batch':>10}")
for ctx in [512, 2048, 8192, 32768]:
    per = kv_cache_gb(ctx)
    print(f"{ctx:>8} {per:>7.2f}GB {int(GPU_KV_BUDGET_GB/per):>9}")
print(f"\n-> with ~{GPU_KV_BUDGET_GB}GB for cache: long contexts collapse the batch, so throughput drops. "
      "Context length is a capacity decision, not just a quality one.")

## 杠杆 3 —— 量化（少搬内存）

既然推理受内存带宽限制，用**更少的 bit** 存权重（以及 cache）就少搬内存，于是既*更快*又*更小*。用 8-bit（INT8/FP8）或 4-bit（例如 AWQ、GPTQ）服务，相比 fp16 大致把显存占用和带宽减半或减到四分之一：

- **让更大的模型放进更小的 GPU：** 4-bit 的 13B 模型能塞进 fp16 塞不下的地方。
- **给 KV cache 腾出显存** → 更大的 batch → 更高吞吐（与杠杆 1–2 叠加）。
- **付出一些质量代价**，8-bit 通常很小，4-bit 更明显。这正是你的**第 04 节评测骨架**派上用场的地方：量化是一次变更，所以要*度量*它，不能靠眼睛看。

> **💡 为什么这几乎是白捡的 ——** 因为瓶颈在内存搬运而不是算术精度，降 bit 常常几乎不损质量，却直接换来速度和容量。它通常是批处理之后的第一个旋钮。

## 综合起来 —— 给一个部署定容量

面试问题很少是「解释一下 PagedAttention」。而是*「你预计 50 请求/秒，需要多少张 GPU？」* 这是用上面那些杠杆搭出来的算术。要能当场算：

In [ ]:
def gpus_needed(target_req_per_s, tokens_per_req, tok_per_s_per_gpu, util=0.7):
    """给一个机队定容量。tok_per_s_per_gpu 是你实测的批处理吞吐（杠杆 1）；
    util<1 留出余量，好让 p99 延迟不至于爆炸（杠杆 1 的取舍）。
    """
    demand_tok_s = target_req_per_s * tokens_per_req
    effective = tok_per_s_per_gpu * util
    import math
    return demand_tok_s, math.ceil(demand_tok_s / effective)

# 场景：50 req/s，每个 300 token，实测一张 GPU 约 2500 批处理 tok/s。
demand, n = gpus_needed(target_req_per_s=50, tokens_per_req=300, tok_per_s_per_gpu=2500)
print(f"demand: {demand:.0f} tok/s")
print(f"GPUs needed (70% util for headroom): {n}")
print("\nlevers that change this number:")
print("  - bigger batch / PagedAttention -> higher tok/s/gpu -> fewer GPUs")
print("  - quantization -> higher tok/s/gpu AND cheaper GPUs")
print("  - longer contexts -> smaller batch -> lower tok/s/gpu -> MORE GPUs")

> **🔵 面试信号 ——** 把这四个动作带进考场：(1) 复述为*延迟还是吞吐*，(2) *需求 = req/s × tokens/req*，(3) *GPU 数 = 需求 / (每 GPU 吞吐 × 利用率)*，(4) 点出能改变每 GPU 吞吐的那个杠杆（批处理、量化、上下文长度）。这个结构*就是* ML 系统设计的答案，在 [第 10 节](../10-ml-system-design/01-designing-an-inference-service.zh.ipynb) 有完整推演。

## 最佳实践与反模式

| ✅ 该做 | ❌ 反模式 |
|---|---|
| 优化前先说明是延迟**还是**吞吐（以及交互式还是批处理） | 没有指标就说「让它更快」，然后把劲用错了地方 |
| 把延迟拆成 TTFT（prefill）和 TPOT（decode） | 只报一个「延迟」数字，把用户真正感受到的 TTFT 藏起来 |
| 把 **KV cache**、而不是算力，当作 batch 的上限 | 明明是 cache 显存不够，却去买算力更大的 GPU |
| 把上下文长度当作*容量*决策（它会缩小 batch） | 为了「保险」把 max context 设成巨大的值，然后纳闷吞吐为什么崩了 |
| 上线前用第 04 节的评测度量量化后的质量 | 因为更快就上 4-bit；然后在生产里发现质量回归 |
| 用 需求 ÷（每 GPU 吞吐 × 利用率）来定机队规模，并留出余量 | 让 GPU 跑在 100% 利用率，然后在流量尖峰下眼看 p99 延迟炸掉 |

## 可选附录 —— 实测一条真实的批处理曲线（需要 GPU）

> **⚠️ 这个附录需要 GPU runtime，可以跳过。** 在 Colab 里：**Runtime → Change runtime type → T4 GPU**。如果在 CPU 上，读一读往下走即可；上面的数学才是要考察的内容。

这个附录是**自包含**的：每个 Colab notebook 都是自己的 runtime，所以它会搭起*它自己的* vLLM 服务器（你没法复用第 01 篇的那个）。启动方式和 [第 01 篇的附录](01-serving-frameworks.zh.ipynb) 相同，然后它用同样的 prompt 逐级提高并发，让你能*亲眼看到*杠杆 1：tokens/sec 随 batch 上升，然后在 KV cache 饱和时变平。

In [ ]:
# 附录第 1 格 —— GPU 检查。如果这里打印出 "no GPU"，就到此为止。
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(gpu.stdout.strip() or "no GPU — the batching math above is what's graded; skip this appendix")

In [ ]:
# 附录第 2 格 —— 安装 vLLM + 修掉 torchaudio 的 CUDA 版本冲突（原因见第 01 篇）。
# vLLM 会升级 PyTorch；Colab 里那份过期的 torchaudio 随后会 import 失败，进而让服务器崩溃。
%pip install -q vllm
!pip uninstall -y torchaudio 2>/dev/null || true
!python -c "import transformers, vllm; print('imports OK — vllm', vllm.__version__)"
# 在全新的 runtime 上不需要重启。如果上面那行报错（pip 升级了某个已加载的
# 包），就 Runtime -> Restart session，然后从 GPU 检查那一格重新往下跑，跳过这个 %pip 行。

In [ ]:
# 附录第 3 格 —— 启动本 notebook 自己的 vLLM 服务器（后台运行，日志写入文件）。
#   --enforce-eager：T4 启动更快更稳   --gpu-memory-utilization 0.85：留出余量
import subprocess, time, urllib.request

LOG = "/content/vllm.log"
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
     "--max-model-len", "2048", "--port", "8000",
     "--enforce-eager", "--gpu-memory-utilization", "0.85"],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

up = False
for _ in range(90):                       # 最多约 7.5 分钟（模型下载 + 加载）
    if server.poll() is not None:
        print(f"server exited (code {server.returncode}). Log tail:\n")
        print("".join(open(LOG).readlines()[-25:])); break
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2)
        print("vLLM server is up on :8000"); up = True; break
    except Exception:
        time.sleep(5)
if not up and server.poll() is None:
    print("still starting. Log tail:\n"); print("".join(open(LOG).readlines()[-15:]))

In [ ]:
# 附录第 4 格 —— 批处理曲线。并发发起 N 个请求，测量聚合 tokens/sec。
# 注意："server is up"（HTTP 层）可能早于引擎预热，所以第一个请求可能和它抢跑
# —— 我们在测量前重试一次，如果服务器确实死了就打印日志尾部后退出。
import time, concurrent.futures
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
PROMPT = "Explain what a KV cache is in two sentences."

def one_call(_=None):
    r = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": PROMPT}], max_tokens=128)
    return r.usage.completion_tokens

# 预热 / 等引擎真正开始接受请求（处理启动期的抢跑）。
ready = False
for attempt in range(1, 13):
    if server.poll() is not None:
        print(f"server process exited (code {server.returncode}). Log tail:\n")
        print("".join(open("/content/vllm.log").readlines()[-25:])); break
    try:
        one_call(); ready = True; break
    except Exception as e:
        print(f"warm-up attempt {attempt}: not ready ({type(e).__name__}) — retrying in 5s"); time.sleep(5)

if ready:
    print(f"\n{'concurrency':>11} {'tok/s':>8}")
    for c in [1, 2, 4, 8, 16]:
        start = time.time()
        with concurrent.futures.ThreadPoolExecutor(max_workers=c) as ex:
            toks = sum(ex.map(one_call, range(c)))
        dt = time.time() - start
        print(f"{c:>11} {toks/dt:>8.0f}")
    print("\n-> tok/s should rise with concurrency (continuous batching), then flatten as the KV cache fills.")
else:
    print("\nserver never became ready — rerun cell 3, then this cell.")

In [ ]:
# 附录第 5 格 —— 释放 GPU。同一时刻只能有一个 vLLM 拥有这张卡；残留的进程会导致
# 下次的 "free memory" 错。（释放显存 != 交出 GPU：只要 runtime 还连着，就仍在
# 消耗你的 Colab GPU 配额 —— 真正收工时用 Runtime -> Disconnect and delete runtime。）
server.terminate()
import time; time.sleep(3)
!pkill -9 -f vllm.entrypoints 2>/dev/null || true
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

## 练习

1. **给你的 capstone 定容量。** 为第 12 节的 capstone 挑一个合理的负载（req/s、tokens/req）和一个每 GPU 吞吐数字（在附录里实测，或假设 ~2500 tok/s）。算出 GPU 数量。然后把上下文长度放大 4 倍再算一遍，并用一句话解释为什么这个数字跳了。
2. **延迟还是吞吐？** 对三个产品（实时聊天助手、每夜跑一次的文档摘要任务、IDE 代码补全插件）分别说出哪项指标占主导，以及它隐含的一个服务化选择（batch size、模型大小，或量化）。
3. **cache 就是上限。** 用 `kv_cache_gb` 那一格，找出在 12 GB cache 预算下、仍能批 **8** 个请求的最大上下文长度。给出这个数字，并说明为了让它翻倍你会牺牲什么。
4. **把量化量化。** 描述（不写代码）你会如何用第 04 节的评测骨架来判断 4-bit 服务对 capstone 是否可接受：度量什么、以什么为基线、以及什么阈值会让你继续用 fp16。